# Model 2: PCA + LDA + SVM
Designed by: Marimbay

This notebook follows the project README workflow and documents model iterations clearly:

- load the shared splits from `data/splits/*.csv`
- verify image paths against the local Kaggle dataset folder
- use shared HOG features (`shared_utils.extract_hog_batch`) for fair comparison with Random Forest
- train a baseline, then tune one block at a time (PCA -> LDA -> SVM C)
- keep decisions based on validation macro F1 (imbalanced dataset)
- evaluate test set once with the selected final configuration

Pipeline: `StandardScaler -> PCA -> LDA -> linear SVM`

Because classes are imbalanced, all SVM fits use `class_weight='balanced'`.
The kernel stays linear and we tune only dimensionality and regularization:

1. **PL1 (baseline):** PCA 95% variance, LDA 11, C=1
2. **PL2:** PCA threshold in {90%, 95%, 99%}, with LDA=11 and C=1 fixed
3. **PL3:** LDA components in {2, 5, 8, 11}, with best PCA and C=1 fixed
4. **PL4:** SVM C in {0.1, 1, 10}, with best PCA and best LDA fixed

A compact experiment log is built across PL1-PL4 to show the iteration trail and the final choice.


In [1]:
# Imports
import sys
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score

sys.path.append("..")
from shared_utils import (
    PROJECT_ROOT,
    load_data,
    eval_model,
    save_metrics,
    extract_hog_batch,
)


In [2]:
# Configurations

# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Directory to save figures
FIGURES_DIR = PROJECT_ROOT / "pca_lda" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MAX_LDA = 11  # 12 classes -> at most C-1 LDA dimensions


In [5]:
# Data Loading 
data = load_data()

train_paths = data["train_paths"]
val_paths = data["val_paths"]
test_paths = data["test_paths"]

train_labels = data["train_labels"]
val_labels = data["val_labels"]
test_labels = data["test_labels"]

class_list = data["class_list"]

print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))
print("Classes:", class_list)

print()
print("Example train image:")
print(train_paths[0])
print("Exists:", Path(train_paths[0]).exists())

experiment_log = []


NameError: name 'DATA_DIR' is not defined

## HOG feature extraction


In [ ]:
# Extract HOG for all three splits
X_train = extract_hog_batch(train_paths, "train")
X_val = extract_hog_batch(val_paths, "val")
X_test = extract_hog_batch(test_paths, "test")

# Labels stay as they are
Y_train = train_labels
Y_val = val_labels
Y_test = test_labels

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"HOG feature dimension: {X_train.shape[1]}")


extracting HOG for 10859 train images...


FileNotFoundError: [Errno 2] No such file or directory: np.str_('C:\\Users\\marim\\OneDrive - Queensland University of Technology\\Desktop\\QUT\\Semester 2\\CAB420 Machine Learning\\Week 13\\Waste-Classification-ML\\data\\garbage_classification\\clothes\\clothes3810.jpg')

In [ ]:
def n_pca_from_variance(X_train_scaled, variance_threshold):
    pca_full = PCA(svd_solver="full", random_state=RANDOM_STATE)
    pca_full.fit(X_train_scaled)
    cumulative = np.cumsum(pca_full.explained_variance_ratio_)
    return int(np.searchsorted(cumulative, variance_threshold) + 1)


def fit_pipeline(
    X_train,
    y_train,
    pca_variance=0.95,
    n_lda=MAX_LDA,
    svm_C=1.0,
    pca_n_components=None,
):
    # Fit on train only
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    if pca_n_components is None:
        pca = PCA(n_components=pca_variance, svd_solver="full", random_state=RANDOM_STATE)
        X_pca = pca.fit_transform(X_scaled)
    else:
        pca = PCA(n_components=pca_n_components, svd_solver="full", random_state=RANDOM_STATE)
        X_pca = pca.fit_transform(X_scaled)

    n_lda_use = min(n_lda, X_pca.shape[1], MAX_LDA)
    lda = LDA(n_components=n_lda_use, solver="svd")
    X_lda = lda.fit_transform(X_pca, y_train)

    svm = SVC(
        kernel="linear",
        C=svm_C,
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE,
    )
    svm.fit(X_lda, y_train)

    return {
        "scaler": scaler,
        "pca": pca,
        "lda": lda,
        "svm": svm,
        "n_pca": pca.n_components_,
        "n_lda": n_lda_use,
    }


def transform_lda(pipe, X):
    X_scaled = pipe["scaler"].transform(X)
    X_pca = pipe["pca"].transform(X_scaled)
    return pipe["lda"].transform(X_pca)


def predict(pipe, X):
    return pipe["svm"].predict(transform_lda(pipe, X))


def predict_proba(pipe, X):
    return pipe["svm"].predict_proba(transform_lda(pipe, X))


def val_macro_f1(pipe, X_val, y_val):
    pred = predict(pipe, X_val)
    return f1_score(y_val, pred, average="macro", zero_division=0)


## PL1 - Baseline


In [ ]:
t0 = time.time()
pipe_v1 = fit_pipeline(X_train, Y_train, pca_variance=0.95, n_lda=MAX_LDA, svm_C=1.0)
train_time_v1 = time.time() - t0

# Macro F1 is used because the dataset is imbalanced.
val_f1_v1 = val_macro_f1(pipe_v1, X_val, Y_val)
print(f"PCA dims: {pipe_v1['n_pca']}  LDA dims: {pipe_v1['n_lda']}")
print(f"Val macro F1 (PL1): {val_f1_v1:.4f}")

experiment_log.append({
    "phase": "PL1_baseline",
    "pca": pipe_v1["n_pca"],
    "lda": pipe_v1["n_lda"],
    "C": 1.0,
    "val_macro_f1": val_f1_v1,
    "note": "baseline: PCA95 + LDA11 + C1",
})

# Predictions
train_pred_v1 = predict(pipe_v1, X_train)

t0 = time.time()
test_pred_v1 = predict(pipe_v1, X_test)
inference_time_v1 = time.time() - t0

result_1 = eval_model(
    Y_test,
    test_pred_v1,
    class_list,
    model_name="PL 1: Baseline PCA95 LDA11 C1",
    y_train=Y_train,
    train_pred=train_pred_v1,
    train_time=train_time_v1,
    inference_time=inference_time_v1,
    save_path=FIGURES_DIR / "pca_lda_v1_confusion_matrices.png",
)


## PCA and LDA figures

Fit on train, plotted on validation. Saved under `figures/`.


In [ ]:
# Cumulative variance plot
scaler_plot = StandardScaler()
X_train_scaled = scaler_plot.fit_transform(X_train)

pca_plot = PCA(svd_solver="full", random_state=RANDOM_STATE)
pca_plot.fit(X_train_scaled)
cumulative = np.cumsum(pca_plot.explained_variance_ratio_)
n_pca_v1 = pipe_v1["n_pca"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
ax.axhline(0.95, color="gray", linestyle="--", label="95% threshold")
ax.axvline(n_pca_v1, color="red", linestyle="--", label=f"kept n={n_pca_v1}")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA: cumulative explained variance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pca_cumulative_variance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'pca_cumulative_variance.png'}")

# 2D PCA on validation
X_val_scaled = scaler_plot.transform(X_val)
X_val_pca2 = pca_plot.transform(X_val_scaled)[:, :2]

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_val_pca2[:, 0], X_val_pca2[:, 1], c=Y_val, cmap="tab20", s=8, alpha=0.7)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Validation set in first two principal components")
handles = [
    plt.Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor=scatter.cmap(scatter.norm(i)), markersize=8, label=class_list[i],
    )
    for i in range(len(class_list))
]
ax.legend(handles=handles, loc="best", fontsize=7, ncol=2)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pca_2d_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'pca_2d_scatter.png'}")

# 2D LDA on validation
X_val_lda = transform_lda(pipe_v1, X_val)
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_val_lda[:, 0], X_val_lda[:, 1], c=Y_val, cmap="tab20", s=8, alpha=0.7)
ax.set_xlabel("LDA 1")
ax.set_ylabel("LDA 2")
ax.set_title("Validation set in first two LDA dimensions")
ax.legend(handles=handles, loc="best", fontsize=7, ncol=2)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lda_2d_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'lda_2d_scatter.png'}")

print(f"Shape trail: {X_train.shape[1]} -> {pipe_v1['n_pca']} -> {pipe_v1['n_lda']}")


## PL2 - PCA variance


In [ ]:
# Macro F1 is used because the dataset is imbalanced.
pca_thresholds = [0.90, 0.95, 0.99]
v2_results = []

scaler_v2 = StandardScaler()
X_train_scaled_v2 = scaler_v2.fit_transform(X_train)

for th in pca_thresholds:
    n_pca = n_pca_from_variance(X_train_scaled_v2, th)
    pipe = fit_pipeline(
        X_train,
        Y_train,
        n_lda=MAX_LDA,
        svm_C=1.0,
        pca_n_components=n_pca,
    )
    vf1 = val_macro_f1(pipe, X_val, Y_val)
    v2_results.append({"threshold": th, "n_pca": n_pca, "val_f1": vf1, "pipe": pipe})
    experiment_log.append({
        "phase": "PL2_pca_search",
        "pca": n_pca,
        "lda": MAX_LDA,
        "C": 1.0,
        "val_macro_f1": vf1,
        "note": f"variance threshold={th:.0%}",
    })
    print(f"  PCA {th:.0%} -> n_pca={n_pca:4d}  val macro F1={vf1:.4f}")

best_v2 = max(v2_results, key=lambda r: r["val_f1"])
best_pca_threshold = best_v2["threshold"]
best_n_pca = best_v2["n_pca"]
pipe_v2 = best_v2["pipe"]

print(f"\nBest PL2: threshold={best_pca_threshold:.0%}, n_pca={best_n_pca}, val F1={best_v2['val_f1']:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar([f"{r['threshold']:.0%}" for r in v2_results], [r["val_f1"] for r in v2_results], color="steelblue")
ax.set_ylabel("Validation macro F1")
ax.set_xlabel("PCA variance threshold")
ax.set_title("PL2: PCA variance search")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pca_variance_search.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'pca_variance_search.png'}")


## PL3 - LDA components


In [ ]:
# Macro F1 is used because the dataset is imbalanced.
lda_candidates = [2, 5, 8, MAX_LDA]
v3_results = []

for n_lda in lda_candidates:
    pipe = fit_pipeline(
        X_train,
        Y_train,
        n_lda=n_lda,
        svm_C=1.0,
        pca_n_components=best_n_pca,
    )
    vf1 = val_macro_f1(pipe, X_val, Y_val)
    v3_results.append({"n_lda": pipe["n_lda"], "val_f1": vf1, "pipe": pipe})
    experiment_log.append({
        "phase": "PL3_lda_search",
        "pca": best_n_pca,
        "lda": pipe["n_lda"],
        "C": 1.0,
        "val_macro_f1": vf1,
        "note": "vary LDA n_components",
    })
    print(f"  n_lda={pipe['n_lda']:2d}  val macro F1={vf1:.4f}")

best_v3 = max(v3_results, key=lambda r: r["val_f1"])
best_n_lda = best_v3["n_lda"]
pipe_v3 = best_v3["pipe"]

print(f"\nBest PL3: n_lda={best_n_lda}, val F1={best_v3['val_f1']:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar([str(r["n_lda"]) for r in v3_results], [r["val_f1"] for r in v3_results], color="seagreen")
ax.set_ylabel("Validation macro F1")
ax.set_xlabel("LDA components")
ax.set_title("PL3: LDA n_components search")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lda_n_components_search.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'lda_n_components_search.png'}")


## PL4 - SVM C


In [ ]:
# Macro F1 is used because the dataset is imbalanced.
svm_C_candidates = [0.1, 1.0, 10.0]
v4_results = []

for C in svm_C_candidates:
    pipe = fit_pipeline(
        X_train,
        Y_train,
        n_lda=best_n_lda,
        svm_C=C,
        pca_n_components=best_n_pca,
    )
    vf1 = val_macro_f1(pipe, X_val, Y_val)
    v4_results.append({"C": C, "val_f1": vf1, "pipe": pipe})
    experiment_log.append({
        "phase": "PL4_svmC_search",
        "pca": best_n_pca,
        "lda": best_n_lda,
        "C": C,
        "val_macro_f1": vf1,
        "note": "vary SVM C",
    })
    print(f"  C={C:<5}  val macro F1={vf1:.4f}")

best_v4 = max(v4_results, key=lambda r: r["val_f1"])
best_svm_C = best_v4["C"]
pipe_final = best_v4["pipe"]

print(f"\nBest PL4: C={best_svm_C}, val F1={best_v4['val_f1']:.4f}")
print(f"Selected: PCA n={best_n_pca}, LDA n={best_n_lda}, SVM C={best_svm_C}")

experiment_log_df = pd.DataFrame(experiment_log)
experiment_log_df = experiment_log_df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
display(experiment_log_df)

best_row = experiment_log_df.iloc[0]
print(
    f"Top validation setup so far: phase={best_row['phase']}, "
    f"PCA={best_row['pca']}, LDA={best_row['lda']}, C={best_row['C']}, "
    f"macro F1={best_row['val_macro_f1']:.4f}"
)


## Iteration reflection (before final test)

Interpretation from validation experiments:

- PL1 baseline gives a reference point for all later changes.
- PL2 shows how PCA compression changes the information available to LDA/SVM.
- PL3 checks whether using all 11 LDA components is necessary or if smaller class-separating subspaces work better.
- PL4 tunes SVM margin hardness with C while keeping the best representation fixed.

Final test evaluation is performed once after selecting the strongest validation setup.

## Save results


In [ ]:
t0 = time.time()
pipe_final = fit_pipeline(
    X_train,
    Y_train,
    n_lda=best_n_lda,
    svm_C=best_svm_C,
    pca_n_components=best_n_pca,
)
train_time_final = time.time() - t0

# Predictions
train_pred_final = predict(pipe_final, X_train)

t0 = time.time()
test_pred_final = predict(pipe_final, X_test)
inference_time_final = time.time() - t0

final_name = f"PL 4: PCA n={best_n_pca} LDA n={best_n_lda} C={best_svm_C}"

result_final = eval_model(
    Y_test,
    test_pred_final,
    class_list,
    model_name=final_name,
    y_train=Y_train,
    train_pred=train_pred_final,
    train_time=train_time_final,
    inference_time=inference_time_final,
    save_path=FIGURES_DIR / "pca_lda_confusion_matrices.png",
)

result_final["model_name"] = "PCA + LDA + SVM"
# Save the selected result for final_comparison.ipynb.
save_metrics(result_final, PROJECT_ROOT / "pca_lda" / "pca_lda_metrics.json")


In [ ]:
# Save predictions for final_comparison.ipynb
Y_prob = predict_proba(pipe_final, X_test)

predictions_df = pd.DataFrame({
    "path": test_paths,
    "true_label": Y_test,
    "true_class": [class_list[i] for i in Y_test],
    "pred_label": test_pred_final,
    "pred_class": [class_list[i] for i in test_pred_final],
    "confidence": np.max(Y_prob, axis=1),
    "correct": Y_test == test_pred_final,
})

predictions_path = PROJECT_ROOT / "pca_lda" / "pca_lda_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)
print("Saved:", predictions_path)
display(predictions_df.head())


## Failure Analysis


In [ ]:
from PIL import Image

wrong = predictions_df[~predictions_df["correct"]].copy()
wrong = wrong.sort_values("confidence", ascending=False).head(6)

if len(wrong) == 0:
    print("No misclassifications on the test set.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for ax, (_, row) in zip(axes, wrong.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(
            f"true: {row['true_class']}\npred: {row['pred_class']} ({row['confidence']:.2f})",
            fontsize=9,
        )
        ax.axis("off")
    for ax in axes[len(wrong):]:
        ax.axis("off")
    fig.suptitle("PCA+LDA+SVM: high-confidence errors", fontsize=12)
    plt.tight_layout()
    out = FIGURES_DIR / "pca_lda_failure_cases.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")
